# Piloto de Mapeo de Áreas y Vías Inundables — Ciudad de Tumbes

**Metodología:**
- Detección satelital de inundaciones con radar **Sentinel-1** según la guía oficial **UN-SPIDER** (ONU).
- Análisis de exposición de red vial (**Screening Fase 1 del BID**) en proyección **EPSG:32717 (UTM 17S)**.

## Bloque 1: Instalación de librerías y autenticación en Google Earth Engine

In [ ]:
!pip install -q earthengine-api geemap osmnx geopandas

import ee
import geemap
import osmnx as ox
import geopandas as gpd

# 1. Autenticación con tu cuenta de Google vinculada a Earth Engine
ee.Authenticate()

# 2. Inicializar proyecto (reemplaza 'TU-PROYECTO-GEE' por el ID de tu proyecto en Google Cloud / GEE)
ee.Initialize(project='TU-PROYECTO-GEE')

## Bloque 2: Extracción de huella de inundación en Tumbes (Método UN-SPIDER)

In [ ]:
# 1. Definir área de estudio: Ciudad de Tumbes [min_lon, min_lat, max_lon, max_lat]
aoi = ee.Geometry.Rectangle([-80.49, -3.60, -80.41, -3.53])

# 2. Fechas: Antes (época seca) vs. Después (lluvias extremas / desborde marzo 2023)
antes_inicio, antes_fin = '2023-01-01', '2023-02-25'
despues_inicio, despues_fin = '2023-03-08', '2023-03-25'

# 3. Cargar colección Sentinel-1 GRD (Polarización VV, modo IW, órbita descendente)
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(aoi)
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
      .select('VV'))

# 4. Mosaico y filtro de ruido (Speckle filter de 50 m estándar UN-SPIDER)
antes = s1.filterDate(antes_inicio, antes_fin).mosaic().clip(aoi).focal_mean(50, 'circle', 'meters')
despues = s1.filterDate(despues_inicio, despues_fin).mosaic().clip(aoi).focal_mean(50, 'circle', 'meters')

# 5. Ratio de cambio y umbral oficial UN-SPIDER (1.25)
diferencia = despues.divide(antes)
inundacion_bruta = diferencia.gt(1.25)

# 6. Limpieza: Quitar agua permanente (cauce normal del río Tumbes) y pendientes > 5°
agua_permanente = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('seasonality').gt(10)
pendiente = ee.Terrain.slope(ee.Image('USGS/SRTMGL1_003')).gt(5)

inundacion_limpia = (inundacion_bruta
                     .where(agua_permanente, 0)
                     .where(pendiente, 0)
                     .selfMask())

# 7. Vectorizar en GEE y convertir a GeoDataFrame en memoria
vectores_ee = inundacion_limpia.reduceToVectors(
    geometry=aoi,
    scale=10,
    geometryType='polygon',
    eightConnected=False,
    maxPixels=1e8
)

gdf_inundacion = geemap.ee_to_gdf(vectores_ee)
print(f"Polígonos de inundación extraídos: {len(gdf_inundacion)}")

## Bloque 3: Descarga de red vial (OSM) y cruce de exposición (BID Fase 1)

In [ ]:
# 1. Descargar red vial de Tumbes dentro del mismo polígono (OSMnx >= 2.0 usa bbox=(left, bottom, right, top))
try:
    grafo = ox.graph_from_bbox(bbox=(-80.49, -3.60, -80.41, -3.53), network_type='drive')
except TypeError:
    # Compatibilidad con versiones anteriores de OSMnx
    grafo = ox.graph_from_bbox(north=-3.53, south=-3.60, east=-80.41, west=-80.49, network_type='drive')

_, gdf_vias = ox.graph_to_gdfs(grafo)

# 2. Reproyectar ambas capas a UTM Zona 17S (EPSG:32717 - oficial para Tumbes)
gdf_vias_utm = gdf_vias.to_crs(epsg=32717)
gdf_inund_utm = gdf_inundacion.to_crs(epsg=32717)

# 3. Intersección espacial: obtener solo los tramos de vías inundados
vias_afectadas = gpd.overlay(gdf_vias_utm, gdf_inund_utm, how='intersection')
vias_afectadas['longitud_m'] = vias_afectadas.geometry.length

# 4. Resumen estadístico por jerarquía vial
vias_afectadas['tipo_via'] = vias_afectadas['highway'].astype(str)
resumen = (vias_afectadas.groupby('tipo_via')['longitud_m']
           .sum()
           .reset_index()
           .sort_values(by='longitud_m', ascending=False))
resumen['longitud_km'] = (resumen['longitud_m'] / 1000).round(2)

print("\n--- KILÓMETROS DE VÍA EXPUESTOS EN TUMBES ---")
print(resumen[['tipo_via', 'longitud_km']].to_string(index=False))

## Bloque 4: Exportar GeoPackage (.gpkg) consolidado para QGIS y GitHub

In [ ]:
# Guardar un único GeoPackage con dos capas: mancha de inundación y vías expuestas
archivo_salida = "tumbes_inundacion_2023.gpkg"

gdf_inund_utm.to_file(archivo_salida, layer="mancha_inundacion", driver="GPKG")

cols_export = [c for c in ['osmid', 'name', 'tipo_via', 'longitud_m', 'geometry'] if c in vias_afectadas.columns]
vias_afectadas[cols_export].to_file(archivo_salida, layer="vias_expuestas", driver="GPKG")

print(f"Archivo '{archivo_salida}' generado correctamente. Descárgalo desde el panel izquierdo de Colab.")